![](../../img/rnn_.png)

In [1]:
import torch
from torch import nn ,optim
from torch.utils.data import DataLoader, Dataset
import re
import os
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:2'

torch.backends.cudnn.enabled = False   # 关键：禁用 cuDNN 的 LSTM 实现


In [2]:
import re
import os

def preprocess_poems(file_paths):
    """
    读取多个古诗文件，清洗每一行，构建全局词典，并编码为 id 序列。

    参数:
        file_paths: list[str] 或 str，可以是多个文件路径组成的列表，也可以是单个文件路径

    返回:
        id_seqs: list[list[int]]，每首诗对应的 id 序列
        id2word: list[str]，id 到字符的映射（索引即 id）
        word2id: dict[str, int]，字符到 id 的映射
    """
    if isinstance(file_paths, str):
        file_paths = [file_paths]

    # 读取所有文件，清洗每一行，收集诗句列表
    poems = []
    for path in file_paths:
        with open(path, 'r', encoding='utf-8') as f:
            for line in f:
                line = line.strip()                       # 去掉首尾空白
                line = re.sub(r'[^\w\s]', '', line)       # 去掉标点符号（保留汉字、字母、数字、下划线、空白）
                if len(line) > 0:
                    poems.append(list(line))

    # 从所有诗句中收集字符，构建全局词典
    char_set = set()
    for poem in poems:
        char_set.update(poem)

    # 排序保证每次运行结果一致，最后加入 <UNKNOWN> 用于处理未登录字符
    id2word = sorted(char_set) + ["<UNKNOWN>"]
    word2id = {word: idx for idx, word in enumerate(id2word)}

    # 统一编码所有诗句
    unk_id = word2id["<UNKNOWN>"]
    id_seqs = [[word2id.get(ch, unk_id) for ch in poem] for poem in poems]

    return id_seqs, id2word, word2id

In [3]:

file_paths = [
    'data/p5-4.txt',
    'data/p5-8.txt',
    'data/p7-4.txt',
    'data/p7-8.txt',
]

id_seqs, id2word, word2id = preprocess_poems(file_paths)
print(len(id_seqs), len(id2word), len(word2id))

234719 9394 9394


In [4]:
class PoetryDataset(Dataset):
    # 诗的序列化表示，以及序列长度L
    def __init__(self, id_seqs, sep_lens):
        self.sep_lens = sep_lens
        self.id_seqs = id_seqs
        
        # 只存 (诗句索引, 起始位置)，不复制实际数据，节省内存
        self.samples = []
        for seq_idx, seq in enumerate(id_seqs):
            # 将每首诗按 sep_lens 分割成多个子序列
            for i in range(len(seq) - sep_lens):
                # (诗句索引, 起始位置)，后续在 __getitem__ 中按需切片
                self.samples.append((seq_idx, i))

    # 数据集大小
    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        # (诗句索引, 起始位置)
        seq_idx, i = self.samples[idx]
        # 获取原始诗句序列
        seq = self.id_seqs[seq_idx]
        L = self.sep_lens
        # (x, y) 一个二元组 (输入序列, 目标序列)
        x = torch.LongTensor(seq[i:i + L])
        y = torch.LongTensor(seq[i + 1:i + L + 1])
        return x, y

In [5]:
class PoetryRNN(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_layers=1, rnn_type='LSTM'):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.rnn_type = rnn_type
        
        # 动态选择 RNN 类型
        if rnn_type == 'LSTM':
            # LSTM 有 cell state，通常比 RNN 更适合长序列
            self.rnn = nn.LSTM(embedding_dim, hidden_dim, num_layers, batch_first=True)
        elif rnn_type == 'GRU':
            # GRU 参数比 LSTM 少，速度更快，效果也不错
            self.rnn = nn.GRU(embedding_dim, hidden_dim, num_layers, batch_first=True)
        else:
            self.rnn = nn.RNN(embedding_dim, hidden_dim, num_layers, batch_first=True)
        
        self.linear = nn.Linear(hidden_dim, vocab_size)

    def forward(self, input, hx=None):
        embed = self.embedding(input)
        
        if self.rnn_type == 'LSTM':
            out, (hn, cn) = self.rnn(embed, hx)
            out = self.linear(out)              
            return out, (hn, cn)
        else:
            out, hn = self.rnn(embed, hx)
            out = self.linear(out)              
            return out, hn

In [27]:
from tqdm.auto import tqdm

def train(model, dataset, optimizer, epoch_num, batch_size, device):

    model.to(device)
    model.train()
    
    criterion = nn.CrossEntropyLoss()
    # 优化器由外部传入
    
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
    
    for epoch in range(epoch_num):
        train_loss = 0.0 
        
        # 进度条
        train_bar = tqdm(dataloader, desc=f"Epoch {epoch+1}/{epoch_num}")
        
        for x, y in train_bar:
            x, y = x.to(device), y.to(device)
            
  
            optimizer.zero_grad()               
            
            output, _ = model(x)               
            # 展平输出: (batch, seq_len, vocab_size) -> (batch*seq_len, vocab_size)
            # 展平标签: (batch, seq_len) -> (batch*seq_len,)
            loss_value = criterion(output.reshape(-1, output.shape[-1]), y.reshape(-1))
            
            loss_value.backward()     
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)          
            optimizer.step()                   

            
            # 累加总损失（每个 batch 的 loss 乘以 batch 大小）
            train_loss += loss_value.item() * x.shape[0]
            
            # 实时更新进度条
            train_bar.set_postfix(loss=f"{loss_value.item():.4f}")
        
        # 计算当前 epoch 的平均损失
        avg_loss = train_loss / len(dataset)
        print(f"Epoch {epoch+1}/{epoch_num}, Avg Loss: {avg_loss:.4f}")

In [7]:
def generate_poem(model, start_seq, id2word, word2id, line_num=4, line_length=7, temperature=1.0, device=None):
    """
    使用训练好的模型生成古诗。

    参数:
        model: 训练好的 PoetryRNN 模型
        start_seq: str，起始字符序列
        id2word: list[str]，id 到字符的映射
        word2id: dict[str, int]，字符到 id 的映射
        line_num: int，生成的行数
        line_length: int，每行的最大长度
        temperature: float，控制采样的随机性，越大越随机
        device: torch.device，若不指定则使用模型所在的设备

    返回:
        poem: str，格式化后的古诗，每行以标点结尾
    """
    if device is None:
        device = next(model.parameters()).device
    
    model.eval()
    
    # 将起始字符转为 id 列表
    input_ids = [word2id.get(ch, word2id["<UNKNOWN>"]) for ch in start_seq]
    generated_ids = input_ids.copy()
    
    # 计算总目标字符数
    total_chars = line_num * line_length
    # 如果起始序列已经超过或等于总字符数，直接截断并格式化
    if len(generated_ids) >= total_chars:
        generated_ids = generated_ids[:total_chars]
        poem = ''.join(id2word[i] for i in generated_ids)
        lines = [poem[i:i+line_length] for i in range(0, len(poem), line_length)]
        return _format_lines(lines)
    
    # 初始输入张量
    input_tensor = torch.LongTensor(input_ids).unsqueeze(0).to(device)  # [1, len(start_seq)]
    hidden = None  # 隐藏状态
    
    with torch.no_grad():
        # 先运行一次起始序列，获得隐藏状态和输出
        output, hidden = model(input_tensor, hidden)
        
        # 需要生成的新字符数
        num_to_generate = total_chars - len(generated_ids)
        
        for _ in range(num_to_generate):
            # 取最后一个时间步的输出，并应用温度
            logits = output[0, -1] / temperature
            probabilities = torch.softmax(logits, dim=-1)
            next_id = torch.multinomial(probabilities, num_samples=1).item()
            generated_ids.append(next_id)
            
            # 准备下一个时间步的输入（只包含新生成的字符）
            input_tensor = torch.LongTensor([[next_id]]).to(device)
            # 传入 hidden，保持上下文
            output, hidden = model(input_tensor, hidden)
    
    # 将 id 转为字符
    poem_chars = [id2word[i] for i in generated_ids]
    poem = ''.join(poem_chars)
    
    # 按行分割
    lines = [poem[i:i+line_length] for i in range(0, len(poem), line_length)]
    return _format_lines(lines)


def _format_lines(lines):
    """给每行末尾加上标点：非最后一行加逗号，最后一行加句号"""
    formatted = []
    for idx, line in enumerate(lines):
        if idx == len(lines) - 1:
            formatted.append(line + '。')
        else:
            formatted.append(line + '，')
    return '\n'.join(formatted)

In [14]:
checkpoint = torch.load('poetry_data_full.pth')

id_seqs = checkpoint['id_seqs']
id2word = checkpoint['id2word']
word2id = checkpoint['word2id']
sep_lens = checkpoint['sep_lens']

dataset = PoetryDataset(id_seqs, sep_lens=sep_lens)
print(len(dataset))  # 输出数据集大小


C:\Users\Administrator\AppData\Local\Temp\ipykernel_18136\3716668376.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load('poetry_data_full.pth')


5572552


In [13]:
### 数据保存

sep_lens = 14  # 序列切分长度
torch.save({
    'id_seqs': id_seqs,
    'id2word': id2word,
    'word2id': word2id,
    'sep_lens': sep_lens,
    'file_paths': file_paths,   # 记录来源，方便追溯
}, 'poetry_data_full.pth')

print(f"\n 完整数据已保存为 poetry_data_full.pth")
print(f"   - 诗句总数: {len(id_seqs)}")
print(f"   - 词表大小: {len(id2word)}")
print(f"   - 序列切分长度: {sep_lens}")


 完整数据已保存为 poetry_data_full.pth
   - 诗句总数: 234719
   - 词表大小: 9394
   - 序列切分长度: 14


In [8]:
# 定义模型
model = PoetryRNN(vocab_size=len(id2word), embedding_dim=128, hidden_dim=256, num_layers=2)

In [9]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

lr = 0.001
epoch_num = 5
batch_size = 128
print(f"使用设备: {device}")


使用设备: cuda


In [ ]:
#train(model, dataset, lr, epoch_num, batch_size, device)

Epoch 1/5:   0%|          | 0/25747 [00:00<?, ?it/s]

Epoch 1/5, Avg Loss: 5.5985


Epoch 2/5:   0%|          | 0/25747 [00:00<?, ?it/s]

Epoch 2/5, Avg Loss: 5.4021


Epoch 3/5:   0%|          | 0/25747 [00:00<?, ?it/s]

Epoch 3/5, Avg Loss: 5.3390


Epoch 4/5:   0%|          | 0/25747 [00:00<?, ?it/s]

Epoch 4/5, Avg Loss: 5.3037


Epoch 5/5:   0%|          | 0/25747 [00:00<?, ?it/s]

Epoch 5/5, Avg Loss: 5.2811


In [ ]:
import torch

# 保存模型权重（文件名标注了参数：RNN, lr0.001, bs128, epoch5, loss5.2）
torch.save(model.state_dict(), 'poetry_rnn_lr0.001_bs128_ep5_loss5.2.pth')

print("模型已保存！")

模型和词表已保存！


**用LSTM训练**

In [25]:
import torch

# 加载之前保存的完整数据
checkpoint = torch.load('poetry_data_full.pth')
id_seqs = checkpoint['id_seqs']
id2word = checkpoint['id2word']
word2id = checkpoint['word2id']
sep_lens = checkpoint['sep_lens']  

print(f"数据加载完成：{len(id_seqs)} 首诗，词表大小 {len(id2word)}，sep_lens={sep_lens}")

C:\Users\Administrator\AppData\Local\Temp\ipykernel_18136\1874861067.py:4: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load('poetry_data_full.pth')


数据加载完成：234719 首诗，词表大小 9394，sep_lens=14


In [26]:
sep_lens = 14
dataset = PoetryDataset(id_seqs, sep_lens=sep_lens)

from torch.utils.data import Subset
import random
subset_indices = random.sample(range(len(dataset)), int(len(dataset) * 0.5))
small_dataset = Subset(dataset, subset_indices)
print(f"本轮训练集大小: {len(small_dataset)}")

model_lstm = PoetryRNN(
    vocab_size=len(id2word),
    embedding_dim=128,
    hidden_dim=128,
    num_layers=2,
    rnn_type='LSTM'
).to(device)



本轮训练集大小: 2786276


In [29]:
# 优化器只创建一次，放在循环外面
optimizer = optim.Adam(model_lstm.parameters(), lr=0.001)

TOTAL_EPOCHS = 20
SAVE_EVERY = 1
START_EPOCH = 0

for ep in range(TOTAL_EPOCHS):
    current_epoch = START_EPOCH + ep + 1
    print(f"开始训练 Epoch {current_epoch}")

    # 传入同一个 optimizer，b保存Adam 动量
    train(model_lstm, small_dataset, optimizer, epoch_num=1,
          batch_size=256, device=device)

    if current_epoch % SAVE_EVERY == 0:
        checkpoint = {
            'model_state_dict': model_lstm.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'id2word': id2word,
            'word2id': word2id,
            'sep_lens': sep_lens,
            'epoch': current_epoch,
        }
        filename = f'poetry_lstm_ep{current_epoch}.pth'
        torch.save(checkpoint, filename)
        print(f"已保存: {filename}")

print("全部训练完成")

开始训练 Epoch 1


Epoch 1/1:   0%|          | 0/10884 [00:00<?, ?it/s]

Epoch 1/1, Avg Loss: 5.9634
已保存: poetry_lstm_ep1.pth
开始训练 Epoch 2


Epoch 1/1:   0%|          | 0/10884 [00:00<?, ?it/s]

Epoch 1/1, Avg Loss: 5.7954
已保存: poetry_lstm_ep2.pth
开始训练 Epoch 3


...

Epoch 1/1, Avg Loss: 5.4928
已保存: poetry_lstm_ep18.pth
开始训练 Epoch 19


Epoch 1/1:   0%|          | 0/10884 [00:00<?, ?it/s]

Epoch 1/1, Avg Loss: 5.4885
已保存: poetry_lstm_ep19.pth
开始训练 Epoch 20


Epoch 1/1:   0%|          | 0/10884 [00:00<?, ?it/s]

Epoch 1/1, Avg Loss: 5.4845
已保存: poetry_lstm_ep20.pth
全部训练完成


In [ ]:

# 继续训练。
# 示例：加载之前保存的模型权重，继续训练 7 轮
checkpoint = torch.load('poetry_lstm_ep13.pth', weights_only=False)

id2word = checkpoint['id2word']
word2id = checkpoint['word2id']
sep_lens = checkpoint['sep_lens']

model_lstm = PoetryRNN(
    vocab_size=len(id2word),
    embedding_dim=128,
    hidden_dim=128,
    num_layers=2,
    rnn_type='LSTM'
).to(device)
model_lstm.load_state_dict(checkpoint['model_state_dict'])
print(f"已恢复 epoch {checkpoint['epoch']}")

# 从第 14 轮继续，跑 7 轮
START_EPOCH = 13
TOTAL_EPOCHS = 7
SAVE_EVERY = 1

for ep in range(TOTAL_EPOCHS):
    current_epoch = START_EPOCH + ep + 1
    print(f"开始训练 Epoch {current_epoch}")

    train(model_lstm, small_dataset, lr=0.001, epoch_num=1,
          batch_size=128, device=device)

    if current_epoch % SAVE_EVERY == 0:
        checkpoint = {
            'model_state_dict': model_lstm.state_dict(),
            'id2word': id2word,
            'word2id': word2id,
            'sep_lens': sep_lens,
            'epoch': current_epoch,
        }
        filename = f'poetry_lstm_ep{current_epoch}.pth'
        torch.save(checkpoint, filename)
        print(f"已保存: {filename}")

In [30]:
poem = generate_poem(model_lstm, "明月", id2word, word2id, line_num=4, line_length=7, temperature=0.8)
print(poem)

明月上帘栊旧交，
游遍三湘月一片，
烟波海钓舟每到，
青宫归独往一时。
